# Fine-tuning EmbeddingGemma 2 with Unsloth: fix its weak spot, then search your own data

EmbeddingGemma 2 was excellent at photos (86% top-1 on 1,000 Flickr photos) but weak at everyday sounds: it picked the right sound first only about **25%** of the time. This notebook fixes that with a small LoRA fine-tune, checks what else changes, and then fine-tunes the text side on a dataset built from a YouTube channel's own transcripts.

| Part | What we train | What we measure |
|---|---|---|
| A1 | Audio encoder + backbone on ESC-50 (folds 1-4) | Held-out fold 5: does sound search improve? |
| A2 | The same, but 10 sound classes never seen in training | Does it learn to *hear*, or just memorize labels? |
| A3 | Nothing new: reuse A1's adapter | Did photo search, voice search or text search get worse? |
| B | Text backbone on "search my channel" question → passage pairs | Retrieval on 24 videos it never saw, plus a general text benchmark |

**Runtime:** an A100 (Runtime → Change runtime type → A100 GPU). Tested on an A100. A T4 is not recommended: it has no bf16, and this model should not run in fp16.

## 0. Install

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps "transformers @ git+https://github.com/huggingface/transformers@92cd495f2720c064bc78eb2d93e28704c5bce51f" "tokenizers>=0.23.1,<0.24" "safetensors>=0.8.0"
!pip install "huggingface_hub>=1.31.0,<2.0" "sentence-transformers>=6.1.0" torchcodec gTTS

The install cell is Unsloth's own, unchanged. Unsloth must be imported before `transformers` and `sentence-transformers`, so it comes first in the next cell.

In [ ]:
from unsloth import FastSentenceTransformer, is_bf16_supported
import csv, gc, json, os, random, time, urllib.request, zipfile
import numpy as np, torch
from datasets import Dataset
from IPython.display import Audio, HTML, display
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

print(torch.cuda.get_device_name(0), "| bf16:", is_bf16_supported())

def free(*objs):
    for o in objs: del o
    gc.collect(); torch.cuda.empty_cache()

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Part A: teach EmbeddingGemma 2 to hear

**ESC-50** has 2,000 five-second clips of 50 everyday sounds (dog, rain, chainsaw, church bells…), split into 5 folds. We train on folds 1-4 (1,600 clips) and test on fold 5 (400 clips the model never hears during training). Each sound class is turned into a sentence, "the sound of a chainsaw", so classification becomes search: embed the clip, embed the 50 label sentences, pick the closest.

In [ ]:
if not os.path.exists("ESC-50-master"):
    urllib.request.urlretrieve("https://github.com/karoldvl/ESC-50/archive/master.zip", "esc50.zip")
    zipfile.ZipFile("esc50.zip").extractall(); os.remove("esc50.zip")

rows = list(csv.DictReader(open("ESC-50-master/meta/esc50.csv")))
classes = sorted({r["category"] for r in rows})
label = lambda c: "the sound of " + c.replace("_", " ")
clip = lambda r: f"ESC-50-master/audio/{r['filename']}"
train_rows = [r for r in rows if r["fold"] != "5"]
test_rows = [r for r in rows if r["fold"] == "5"]
print(len(classes), "classes |", len(train_rows), "train clips |", len(test_rows), "test clips")

In [ ]:
def load_audio_model():
    return FastSentenceTransformer.from_pretrained(
        model_name = "unsloth/embeddinggemma-2",
        max_seq_length = 512,                   # 5 s of audio = 125 tokens
        config_kwargs = {"vision_config": None}, # text + audio only (570M)
        full_finetuning = False,
    )

def esc50_eval(model, rows = test_rows, prompt_name = "SearchQuery", fmt = label):
    model.eval()
    with torch.no_grad():
        A = model.encode([{"audio": clip(r)} for r in rows], batch_size = 16, convert_to_tensor = True)
        L = model.encode([fmt(c) for c in classes], prompt_name = prompt_name, convert_to_tensor = True)
    rank = model.similarity(A, L).float().cpu().argsort(dim = 1, descending = True)
    y = torch.tensor([classes.index(r["category"]) for r in rows])
    per = {c: round(100 * (rank[y == k, 0] == k).float().mean().item()) for k, c in enumerate(classes) if (y == k).any()}
    return {"top-1": round(100 * (rank[:, 0] == y).float().mean().item(), 1),
            "top-5": round(100 * (rank[:, :5] == y[:, None]).any(1).float().mean().item(), 1)}, per

model = load_audio_model()
before, before_per = esc50_eval(model)
print("baseline, 'the sound of …' + SearchQuery prompt:", before)
print("baseline, bare label + Classification prompt:   ", esc50_eval(model, prompt_name = "Classification", fmt = lambda c: c.replace("_", " "))[0])
print("classes at 0%:", [c for c, v in before_per.items() if v == 0])

Two things to notice before training: about one clip in four is right, and the model's built-in `Classification` prompt does *worse* than writing the label as a search query. So the prompt alone can't fix this: we need training.

## A1. LoRA on the audio encoder and the backbone

`finetune_audio_layers = True` adds LoRA adapters inside the 300M audio encoder as well as the shared backbone, so the model can change *how it hears*, not just how it reads the label. Only ~1-2% of the weights train.

In [ ]:
def add_lora(model):
    return FastSentenceTransformer.get_peft_model(
        model, r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none",
        target_modules = LORA_TARGETS,
        finetune_audio_layers = True,          # also adapt the audio encoder
        use_gradient_checkpointing = "unsloth",
        random_state = 3407, task_type = "FEATURE_EXTRACTION",
    )

def esc50_pairs(rows):
    return Dataset.from_list([{"anchor": label(r["category"]), "positive": clip(r)} for r in rows]).shuffle(seed = 3407)

def train(model, dataset, epochs = 2, lr = 1e-4, batch = 16, prompts = None, out = "outputs"):
    args = SentenceTransformerTrainingArguments(
        output_dir = out, num_train_epochs = epochs, per_device_train_batch_size = batch,
        learning_rate = lr, warmup_ratio = 0.05, lr_scheduler_type = "linear", logging_steps = 10,
        bf16 = is_bf16_supported(), fp16 = not is_bf16_supported(),
        batch_sampler = BatchSamplers.NO_DUPLICATES,  # same label twice in a batch = a false negative
        prompts = prompts, remove_unused_columns = False, save_strategy = "no", report_to = "none", seed = 3407,
    )
    t = time.time()
    SentenceTransformerTrainer(model = model, args = args, train_dataset = dataset,
                               loss = losses.MultipleNegativesRankingLoss(model)).train()
    print(f"trained in {time.time() - t:.0f} s")

model = add_lora(model)
model.print_trainable_parameters() if hasattr(model, "print_trainable_parameters") else None
train(model, esc50_pairs(train_rows), prompts = {"anchor": model.prompts["SearchQuery"]})

Why `BatchSamplers.NO_DUPLICATES`? `MultipleNegativesRankingLoss` treats every *other* pair in the batch as a wrong answer. ESC-50 has 32 training clips per label, so without this sampler two "the sound of rain" pairs could land in one batch and the model would be told that rain is not rain.

In [ ]:
after, after_per = esc50_eval(model)
print("before:", before, "\nafter: ", after)
gains = sorted(((after_per[c] - before_per[c], c) for c in classes), reverse = True)
print("\nbiggest gains:", [(c, f"{before_per[c]}% → {after_per[c]}%") for g, c in gains[:8]])
print("still weakest:", sorted(((after_per[c], c) for c in classes))[:5])
model.save_pretrained("esc50_lora")

In [ ]:
# listen: what does each query find now, among the 400 held-out clips?
model.eval()
with torch.no_grad():
    A = model.encode([{"audio": clip(r)} for r in test_rows], batch_size = 16, convert_to_tensor = True)
for q in ["a chainsaw cutting wood", "a crackling campfire", "a dog barking"]:
    s = model.similarity(model.encode(q, prompt_name = "SearchQuery", convert_to_tensor = True), A)[0]
    i = s.argmax().item()
    print(f"{q!r} → {test_rows[i]['category']} ({s[i]:.3f})"); display(Audio(clip(test_rows[i])))

## A2. Is it learning to hear, or memorizing 50 labels?

A1 tested new clips, but of the same 50 labels it trained on. Now we hide 10 classes completely: train on the other 40, then test fold 5 clips of the 10 hidden classes (still ranked against all 50 labels).

In [ ]:
free(model)
hidden = sorted(random.Random(0).sample(classes, 10))
print("never seen in training:", hidden)
seen_test = [r for r in test_rows if r["category"] not in hidden]
hidden_test = [r for r in test_rows if r["category"] in hidden]

model = load_audio_model()
b_seen, b_hidden = esc50_eval(model, seen_test)[0], esc50_eval(model, hidden_test)[0]
model = add_lora(model)
train(model, esc50_pairs([r for r in train_rows if r["category"] not in hidden]), prompts = {"anchor": model.prompts["SearchQuery"]})
a_seen, a_hidden = esc50_eval(model, seen_test)[0], esc50_eval(model, hidden_test)[0]
print(f"40 trained classes : {b_seen} → {a_seen}")
print(f"10 hidden classes  : {b_hidden} → {a_hidden}")
free(model)

## A3. Did we break anything?

The adapter from A1 also changed the shared backbone and the audio encoder, and those are used by *everything*. So we load the full model (text + vision + audio), with and without the adapter, and re-run three checks:

1. **Photo search**: 5,000 Flickr captions → 1,000 photos (86% top-1 in the first video).
2. **Voice search**: 100 captions spoken with gTTS → the same 1,000 photos.
3. **Text search**: NanoBEIR (four small public retrieval sets).

In [ ]:
from huggingface_hub import hf_hub_download
from PIL import Image
from gtts import gTTS
import librosa, soundfile as sf

REPO = "nlphuji/flickr_1k_test_image_text_retrieval"
if not os.path.exists("images_flickr_1k_test"):
    zipfile.ZipFile(hf_hub_download(REPO, "images_flickr_1k_test.zip", repo_type = "dataset")).extractall(".")
fl = list(csv.DictReader(open(hf_hub_download(REPO, "test_1k_flickr.csv", repo_type = "dataset"))))
photos = [f"images_flickr_1k_test/{r['filename']}" for r in fl]
caps = [c for r in fl for c in json.loads(r["raw"])]
owner = [i for i, r in enumerate(fl) for _ in json.loads(r["raw"])]

os.makedirs("spoken", exist_ok = True)
spoken_idx = list(range(0, len(caps), 50))  # 100 captions, one per 10 photos
for i in spoken_idx:
    p = f"spoken/{i}.wav"
    if not os.path.exists(p):
        gTTS(caps[i]).save("tmp.mp3"); w, _ = librosa.load("tmp.mp3", sr = 16000); sf.write(p, w, 16000)

def r_at(sim, gold, k):
    top = sim.argsort(dim = 1, descending = True)[:, :k]
    return round(100 * (top == torch.tensor(gold)[:, None]).any(1).float().mean().item(), 1)

def side_effects(model):
    from sentence_transformers.sentence_transformer.evaluation import NanoBEIREvaluator
    model.eval()
    # Wrap execution in autocast to handle mixed float16/float32 operations on T4 GPUs safely
    with torch.no_grad(), torch.amp.autocast('cuda'):
        P = model.encode([{"image": p} for p in photos], batch_size = 16, convert_to_tensor = True) # Reduced batch size to save memory on T4
        C = model.encode(caps, prompt_name = "SearchQuery", batch_size = 64, convert_to_tensor = True) # Reduced batch size to save memory on T4
        V = model.encode([{"audio": f"spoken/{i}.wav"} for i in spoken_idx], batch_size = 16, convert_to_tensor = True)
    s = model.similarity(C, P).float().cpu(); v = model.similarity(V, P).float().cpu()
    nano = NanoBEIREvaluator(dataset_names = ["nfcorpus", "scifact", "fiqa2018", "arguana"],
                             query_prompts = model.prompts["query"], corpus_prompts = model.prompts["document"],
                             batch_size = 32, show_progress_bar = False)(model) # Reduced batch size
    return {"photo R@1": r_at(s, owner, 1), "voice→photo R@1": r_at(v, [owner[i] for i in spoken_idx], 1),
            "voice→photo R@5": r_at(v, [owner[i] for i in spoken_idx], 5),
            "NanoBEIR nDCG@10": round(100 * nano["NanoBEIR_mean_cosine_ndcg@10"], 1)}

# Force garbage collection before we start loading the heavy models
free()

# 1. Load and evaluate full baseline model
full = FastSentenceTransformer.from_pretrained("unsloth/embeddinggemma-2", for_inference = True)
base_fx = side_effects(full)
processor = full[0].processor
del full
free()

# 2. Load and evaluate tuned model
tuned = FastSentenceTransformer.from_pretrained("esc50_lora")
tuned[0].processor = processor
tuned_fx = side_effects(tuned)
del tuned
free()

for k in base_fx: print(f"{k:18s} before {base_fx[k]:5}   after {tuned_fx[k]:5}")

# Part B: search your own text

Now the text side. The dataset is **SQuAD**: Wikipedia paragraphs, each with a question written by a person ("Which NFL team represented the AFC at Super Bowl 50?"). We keep one question per paragraph (about 2,000 passages). The split is **by article**: about 80% of the articles for training and 20% the model never sees. The task is "find the paragraph that answers this question" among *all* passages.

To use your own data, build `data` as a list of `{"chunk_id", "query", "title", "text", "split"}` records instead.

In [ ]:
from datasets import load_dataset

# SQuAD: Wikipedia paragraphs with crowd-written questions. Keep one question per paragraph.
seen, data = set(), []
for r in load_dataset("rajpurkar/squad", split = "validation"):
    if r["context"] in seen: continue
    seen.add(r["context"])
    data.append({"chunk_id": f"c{len(data)}", "query": r["question"], "title": r["title"].replace("_", " "), "text": r["context"]})

# split by article, so test questions come from articles the model never sees
titles = sorted({d["title"] for d in data}); random.Random(0).shuffle(titles)
test_titles = set(titles[: len(titles) // 5])
for d in data: d["split"] = "test" if d["title"] in test_titles else "train"

doc = lambda d: f"title: {d['title']} | text: {d['text']}"   # titled-document format from the model card
train_d = [d for d in data if d["split"] == "train"]; test_d = [d for d in data if d["split"] == "test"]
print(len(data), "passages |", len(train_d), "train |", len(test_d), "test (from", len(test_titles), "unseen articles)")
print("example:", test_d[0]["query"])

In [ ]:
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator, NanoBEIREvaluator

model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/embeddinggemma-2",
    max_seq_length = 512,
    config_kwargs = {"vision_config": None, "audio_config": None},  # text only: 270M
    full_finetuning = False,
)
channel_eval = InformationRetrievalEvaluator(
    queries = {d["chunk_id"]: d["query"] for d in test_d},
    corpus = {d["chunk_id"]: doc(d) for d in data},          # search all passages
    relevant_docs = {d["chunk_id"]: [d["chunk_id"]] for d in test_d},
    query_prompt = model.prompts["SearchQuery"], name = "channel",
    accuracy_at_k = [1, 5, 10], batch_size = 64, show_progress_bar = False,
)
nano = NanoBEIREvaluator(dataset_names = ["nfcorpus", "scifact", "fiqa2018", "arguana"],
                         query_prompts = model.prompts["query"], corpus_prompts = model.prompts["document"],
                         batch_size = 64, show_progress_bar = False)
def text_scores(model):
    c = channel_eval(model); n = nano(model)
    return {"top-1": round(100 * c["channel_cosine_accuracy@1"], 1), "top-10": round(100 * c["channel_cosine_accuracy@10"], 1),
            "nDCG@10": round(100 * c["channel_cosine_ndcg@10"], 1), "NanoBEIR": round(100 * n["NanoBEIR_mean_cosine_ndcg@10"], 1)}
text_before = text_scores(model); print(text_before)

In [ ]:
model = FastSentenceTransformer.get_peft_model(
    model, r = 32, lora_alpha = 64, lora_dropout = 0, bias = "none",
    target_modules = LORA_TARGETS, use_gradient_checkpointing = "unsloth",
    random_state = 3407, task_type = "FEATURE_EXTRACTION",
)
pairs = Dataset.from_list([{"query": d["query"], "passage": doc(d)} for d in train_d]).shuffle(seed = 3407)
train(model, pairs, epochs = 3, lr = 5e-5, batch = 32, prompts = {"query": model.prompts["SearchQuery"]}, out = "outputs_text")
text_after = text_scores(model)
for k in text_before: print(f"{k:10s} before {text_before[k]:5}   after {text_after[k]:5}")
model.save_pretrained("channel_lora")

In [ ]:
# try it: questions from articles the model never trained on
model.eval()
with torch.no_grad():
    D = model.encode([doc(d) for d in data], batch_size = 64, convert_to_tensor = True)
def ask(q, k = 3):
    s = model.similarity(model.encode(q, prompt_name = "SearchQuery", convert_to_tensor = True), D)[0]
    for i in s.topk(k).indices.tolist():
        d = data[i]
        print(f"{s[i]:.3f}  [{d['title']}]  {d['text'][:100]}…")
for d in [test_d[5], test_d[120], test_d[-1]]:
    print("Q:", d["query"], "\n   correct article:", d["title"]); ask(d["query"]); print()

## Rules of thumb
- Measure first, on data the model never trains on (held-out clips, held-out videos).
- Pairs + `MultipleNegativesRankingLoss` + `BatchSamplers.NO_DUPLICATES` is the whole recipe; the other items in the batch are your negatives.
- Use the model's prompts consistently in training and search (`SearchQuery` for queries, a titled `title: … | text: …` document).
- Check what else your adapter touches: the backbone is shared by every modality.
- Save the LoRA adapter (tens of MB), not a full copy of the model.